In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent / "src" / "data_loading"))


from phhs import load_phhs_file, count_games_from_phhs

In [2]:
DATA_PATH = Path().resolve().parent / "data" / "phhs"

First it is necessary to filter games which are No-limit Texas hold ‘em (`"NT"`) and are Head's Up (2 players). We save all paths to files that have such games in variable `paths_with_desired_games`.

In [3]:
allowed = {".phhs", ".phh"}
number_of_games = 0
paths_with_desired_games = []

all_paths = [p for p in DATA_PATH.rglob("*") if p.is_file()]
num_of_all_files = len(all_paths)

for i, path in enumerate(all_paths, start = 1):
    percent = round(i / num_of_all_files * 100, 2)

    if i % 100 == 0:
        print(f"\rProgress: {i}/{num_of_all_files}. Percent of visited files: {percent}%. Founded games: {number_of_games}", end = "", flush = True)
    if (path.suffix in allowed):
        founded_games_in_the_path = count_games_from_phhs(path)
        if founded_games_in_the_path > 0:
            paths_with_desired_games.append(path)
            number_of_games += founded_games_in_the_path
#    if i > 500:
#        break

Progress: 31800/31870. Percent of visited files: 99.78%. Founded games: 3051624

Next we load all the games from paths saved in `paths_with_desired_games` to dictionaries. We save games in variable `games_list` and actions for each game in the variable `actions_list`.

In [ ]:
games_list = []
actions_list = []

num_of_all_files = len(paths_with_desired_games)

for i, path in enumerate(paths_with_desired_games, start = 1):
    percent = round(i / num_of_all_files * 100, 2)
    
    print(f"\rProgress: {i}/{num_of_all_files}. Percent of visited files: {percent}%", end = "", flush = True)
    game, actions = load_phhs_file(path)
    games_list += game
    actions_list += actions
    
#    if i > 100:
#        break

Progress: 3501/21191. Percent of visited files: 16.52%

In [ ]:
games_list

Now we go through loaded games in `games_list` and `actions_list` and choose only No-limit Texas hold‘em Head's Up games and their actions and save them in `desirable_games` and `desirable_actions`. 

In [ ]:
desirable_games = []
desirable_actions = []
for game, actions in zip(games_list, actions_list):

    if game["variant"] != "NT":
        continue

    num_of_players = sum("starting_stack" in key for key in game)
    if num_of_players != 2:
        continue
        
    has_unknown_cards = any(
        ("??" in action.get('hand_card_1')) or ("??" in action.get('hand_card_2'))
        for action in actions
        if (('hand_card_1' in action) and ('hand_card_2' in action))
    )
    if has_unknown_cards:
        continue

    desirable_games.append(game)
    desirable_actions.append(actions)

#with open("output.txt", "w") as f:
#    print(f"Liczba wykrytych gier: {len(desirable_games)}. Gry: {desirable_games}.", file = f )


In [ ]:
len(desirable_games) == len(desirable_actions) #checking if everyting is right

In [ ]:
desirable_actions

We convert data into data frames. As a result we get two data frames: `df_all_games` with information about the games and `df_all_actions` with information about the actions. 

In [ ]:
import pandas as pd

columns_order_games = ['game_id', 'variant', 'small_blind', 'big_blind', 'min_bet', 'starting_stack_1', 'starting_stack_2']
columns_order_actions = ['game_id', 'action_id', 'actor', 'action', 'target', 'hand_card_1', 'hand_card_1', 'community_card_1','community_card_2', 'community_card_3', 'community_card_4', 'community_card_5', 'cbr_amount']
df_all_games = pd.DataFrame(columns = columns_order_games)
df_all_actions = pd.DataFrame(columns = columns_order_actions)

for i, (game, actions) in enumerate(zip(desirable_games, desirable_actions)):

    game["game_id"] = i
    df_all_games.loc[len(df_all_games)] = game

    for action in actions:
        action["game_id"] = i
        df_all_actions.loc[len(df_all_actions)] = action
        

In [ ]:
df_all_games

In [ ]:
df_all_actions.loc[df_all_actions["game_id"] == 15,:]

In [ ]:
#exporting data frames
df_all_games.to_csv("data_games_phhs.csv", index = False) 
df_all_actions.to_csv("data_actions_phhs.csv", index = False)